# Home-Action Detection - Skeleton ST-GCN Fine-Tuning (Colab)

Multi-class indoor action recognition over **skeletons** (background-invariant).
Replaces the dual-head R3D model. Classes are discovered automatically from your
dataset folders, so it trains 7 or 10 classes depending on what you upload.

Target classes (IndoorActionDataset, full 10-class set):
  falling_down, lying_on_the_floor, sitting_down, standing_up, walking,
  watching_tv, no_action, cleaning, eating, blowing_nose_or_sneezing

The two **alert** classes (falling_down, lying_on_the_floor) are the
safety-critical ones; the rest enrich the scene description.

**Tuned for free Colab T4 + 4 GB local inference** (yolo11s-pose, single-person
skeletons, clip caps). Phase 1 extraction is resume-safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
#  CONFIG  -  after you upload the whole Fine_tuning folder to
#  MyDrive/VisionAI/, these paths already point at this model folder.
#  (datasets live INSIDE it - see DATASETS.txt.)  "" = skip a source.
# =====================================================================
BASE = "/content/drive/MyDrive/VisionAI/Fine_tuning/02_home_action_stgcn"

# Primary dataset: IndoorActionDataset (has train/ validation/ test/ splits,
# each with one subfolder per class). Class folder names may contain spaces.
INDOOR_ROOT = BASE + "/IndoorActionDataset-video"

# OPTIONAL extra data to boost weak classes (esp. falls). Layout: <root>/<class>/*.mp4
# Put fall clips in a folder literally named "falling down" (or "lying on the floor"),
# so they map onto the same classes. Auto-split into train/val. "" to skip.
EXTRA_ROOT  = BASE + "/home_action_extra"

# Working dir: skeleton cache + trained checkpoint
WORK_DIR    = BASE + "/_work"

# =====================================================================
#  HYPERPARAMETERS
# =====================================================================
POSE_MODEL    = "yolo11s-pose.pt"   # MUST match the app's deployed pose model
POSE_CONF     = 0.30
CLIP_LEN      = 32
MAX_PERSONS   = 1                    # home actions = per-subject; classify each person
IMG_NORM      = True

MAX_CLIPS_PER_CLASS = 0             # 0 = use all (dataset is small); cap if you add lots of extra
BATCH_SIZE    = 64
EPOCHS        = 80                   # max per fold; early stopping ends sooner
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.20                 # only for EXTRA_ROOT (INDOOR_ROOT has its own split)
SEED          = 42

import os
os.makedirs(WORK_DIR, exist_ok=True)
CACHE_DIR = os.path.join(WORK_DIR, "skeleton_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
print("Work dir :", WORK_DIR)

In [ ]:
!pip -q install ultralytics tqdm
import torch, subprocess
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

## Phase 1 - Discover classes + extract skeletons
Classes are auto-discovered from INDOOR_ROOT/train (and EXTRA_ROOT). Folder names
are normalized: "falling down" -> "falling_down". Resume-safe caching.

In [ ]:
import os, cv2, json, hashlib, numpy as np, random
from ultralytics import YOLO
from tqdm import tqdm

random.seed(SEED); np.random.seed(SEED)
USE_HALF = torch.cuda.is_available()
pose_model = YOLO(POSE_MODEL)
VIDEO_EXT = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

def norm_label(name):
    return name.strip().lower().replace(' ', '_')

def discover_classes():
    names = set()
    roots = []
    if INDOOR_ROOT: roots.append(os.path.join(INDOOR_ROOT, 'train'))
    if EXTRA_ROOT:  roots.append(EXTRA_ROOT)
    for r in roots:
        if os.path.isdir(r):
            for d in os.listdir(r):
                if os.path.isdir(os.path.join(r, d)):
                    names.add(norm_label(d))
    return sorted(names)

CLASS_NAMES = discover_classes()
LABEL2IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
assert CLASS_NAMES, "No class folders found - check INDOOR_ROOT / EXTRA_ROOT."
print("Discovered %d classes:" % len(CLASS_NAMES), CLASS_NAMES)

def _sample_indices(n_total, n):
    if n_total <= 0: return []
    if n_total >= n: return [int(v) for v in np.linspace(0, n_total - 1, n)]
    return list(range(n_total)) + [n_total - 1] * (n - n_total)

def extract_skeleton(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1.0
    H = cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1.0
    if total <= 0:
        cap.release(); return None
    idx_list = _sample_indices(total, CLIP_LEN)
    need = set(idx_list); grabbed = {}; fi = 0; maxneed = max(idx_list)
    while fi <= maxneed:
        ret, fr = cap.read()
        if not ret: break
        if fi in need: grabbed[fi] = fr
        fi += 1
    cap.release()
    if not grabbed: return None
    frames, last = [], None
    for fidx in idx_list:
        fr = grabbed.get(fidx, last)
        if fr is None: fr = next(iter(grabbed.values()))
        frames.append(fr); last = fr
    results = pose_model.predict(frames, conf=POSE_CONF, classes=[0], verbose=False, half=USE_HALF)
    clip = np.zeros((CLIP_LEN, MAX_PERSONS, 17, 3), dtype=np.float32)
    for t, res in enumerate(results):
        if res.keypoints is None or res.keypoints.data is None: continue
        kp = res.keypoints.data.cpu().numpy()
        if kp.shape[0] == 0: continue
        order = np.argsort(-kp[:, :, 2].mean(axis=1))[:MAX_PERSONS]
        for m, pidx in enumerate(order):
            person = kp[pidx].copy()
            if IMG_NORM:
                person[:, 0] = (person[:, 0] / W - 0.5) * 2.0
                person[:, 1] = (person[:, 1] / H - 0.5) * 2.0
            clip[t, m] = person
    return clip

def cache_path(video_path):
    h = hashlib.md5(video_path.encode()).hexdigest()[:16]
    base = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(CACHE_DIR, base + "_" + h + ".npy")

def _list_videos(d):
    out = []
    for root, _, files in os.walk(d):
        for f in files:
            if f.lower().endswith(VIDEO_EXT):
                out.append(os.path.join(root, f))
    return out

def _cap(lst):
    random.shuffle(lst)
    return lst if MAX_CLIPS_PER_CLASS in (0, None) else lst[:MAX_CLIPS_PER_CLASS]

def gather_samples():
    samples = []   # (video_path, label_idx, split)
    if INDOOR_ROOT:
        for split_name, tag in (('train', 'train'), ('validation', 'val'), ('test', 'test')):
            base = os.path.join(INDOOR_ROOT, split_name)
            if not os.path.isdir(base): continue
            for d in os.listdir(base):
                cd = os.path.join(base, d)
                if not os.path.isdir(cd): continue
                lab = LABEL2IDX.get(norm_label(d))
                if lab is None: continue
                for f in _cap(_list_videos(cd)):
                    samples.append((f, lab, tag))
    if EXTRA_ROOT and os.path.isdir(EXTRA_ROOT):
        for d in os.listdir(EXTRA_ROOT):
            cd = os.path.join(EXTRA_ROOT, d)
            if not os.path.isdir(cd): continue
            lab = LABEL2IDX.get(norm_label(d))
            if lab is None: continue
            for f in _cap(_list_videos(cd)):
                samples.append((f, lab, ''))
    return samples

samples = gather_samples()
print("Source clips selected:", len(samples))
assert samples, "No clips found - check dataset paths."

manifest = []
for vpath, lab, split in tqdm(samples, desc="Extracting skeletons"):
    cpath = cache_path(vpath)
    if not os.path.exists(cpath):
        clip = extract_skeleton(vpath)
        if clip is None: continue
        np.save(cpath, clip)
    manifest.append({"npy": cpath, "label": lab, "split": split})

with open(os.path.join(WORK_DIR, "manifest.json"), "w") as f:
    json.dump({"class_names": CLASS_NAMES, "items": manifest}, f)
print("Cached samples:", len(manifest))

## Phase 2 - Datasets / loaders (multi-class, class-balanced)

In [ ]:
import json, numpy as np, torch, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

blob = json.load(open(os.path.join(WORK_DIR, "manifest.json")))
CLASS_NAMES = blob["class_names"]; manifest = blob["items"]
NUM_CLASSES = len(CLASS_NAMES)
random.seed(SEED); np.random.seed(SEED)

# Pool train+validation+test clips, then a stratified 75/25 split into a
# train+val pool (K-fold CV runs here) and a held-out TEST set (scored once).
all_items  = list(manifest)
all_labels = np.array([m["label"] for m in all_items])
trainval_items, test_items = train_test_split(
    all_items, test_size=0.25, random_state=SEED, stratify=all_labels)
print("Classes:", NUM_CLASSES, "| pool:", len(all_items),
      "| train+val:", len(trainval_items), " held-out test:", len(test_items))

class SkeletonDataset(Dataset):
    def __init__(self, items, augment=False):
        self.items = items; self.augment = augment
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        m = self.items[i]
        x = np.load(m["npy"]).astype(np.float32)
        if self.augment:
            if random.random() < 0.5: x[..., 0] *= -1.0
            x[..., :2] += np.random.normal(0, 0.01, x[..., :2].shape).astype(np.float32)
            x[..., :2] *= np.random.uniform(0.9, 1.1)        # scale jitter
        x = torch.from_numpy(x).permute(3, 0, 2, 1).contiguous()
        return x, m["label"]

def make_loaders(tr_items, va_items):
    tr_ds = SkeletonDataset(tr_items, augment=True)
    va_ds = SkeletonDataset(va_items, augment=False)
    labels = np.array([m["label"] for m in tr_items])
    cls_count = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float32)
    cls_w = 1.0 / np.maximum(cls_count, 1.0)
    samp_w = [cls_w[l] for l in labels]
    sampler = WeightedRandomSampler(samp_w, num_samples=len(tr_items), replacement=True)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, va_loader, cls_w

_cnt = np.bincount(np.array([m["label"] for m in trainval_items]), minlength=NUM_CLASSES).astype(int)
for c, n in zip(CLASS_NAMES, _cnt): print("  %-26s %d" % (c, n))

## ST-GCN model (COCO-17 graph, multi-class head)

In [ ]:
import torch, torch.nn as nn, numpy as np
COCO_EDGES = [(0,1),(0,2),(1,3),(2,4),(0,5),(0,6),(5,6),(5,7),(7,9),
              (6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
NUM_JOINTS = 17
def build_adjacency():
    A = np.zeros((NUM_JOINTS, NUM_JOINTS), dtype=np.float32)
    for i, j in COCO_EDGES: A[i, j] = 1.0; A[j, i] = 1.0
    A += np.eye(NUM_JOINTS, dtype=np.float32)
    deg = A.sum(1); Dinv = np.diag(1.0 / np.maximum(deg, 1e-6)).astype(np.float32)
    return (Dinv @ A).astype(np.float32)

class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True):
        super().__init__()
        self.register_buffer("A", torch.tensor(A))
        self.edge_imp = nn.Parameter(torch.ones_like(self.A))
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.tcn = nn.Sequential(nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)), nn.BatchNorm2d(cout))
        if not residual: self.res = None
        elif cin == cout and stride == 1: self.res = nn.Identity()
        else: self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        res = 0 if self.res is None else self.res(x)
        x = self.gcn(x)
        x = torch.einsum("nctv,vw->nctw", x, self.A * self.edge_imp)
        x = self.tcn(x)
        return self.relu(x + res)

class STGCN(nn.Module):
    def __init__(self, in_ch=3, num_classes=10):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(in_ch * NUM_JOINTS)
        self.layers = nn.ModuleList([
            STGCNBlock(in_ch, 64, A, residual=False), STGCNBlock(64, 64, A),
            STGCNBlock(64, 128, A, stride=2), STGCNBlock(128, 128, A),
            STGCNBlock(128, 256, A, stride=2), STGCNBlock(256, 256, A)])
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 1, 3, 2).contiguous().view(N * M, C * V, T)
        x = self.data_bn(x)
        x = x.view(N * M, C, V, T).permute(0, 1, 3, 2).contiguous()
        for layer in self.layers: x = layer(x)
        x = x.mean(dim=[2, 3]).view(N, M, -1).mean(dim=1)
        return self.fc(self.drop(x))

_m = STGCN(num_classes=NUM_CLASSES)
_dummy = torch.zeros(2, 3, CLIP_LEN, NUM_JOINTS, MAX_PERSONS)
print("Model OK, output shape:", tuple(_m(_dummy).shape))

## Train - Stratified 5-Fold CV (best model by val_loss)
Stratified K-Fold on the 75% train+val pool keeps every class balanced per fold -
the fix for the noisy single-split validation that made the old run look like it
was overfitting. Each fold checkpoints on the **lowest val_loss** and early-stops
on patience; the best-val_loss fold is saved. Class weighting + label smoothing +
skeleton augmentation regularize; the fall classes stay the priority.

In [ ]:
import os, copy, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SPLITS = 5            # Stratified K-Fold on the 75% train+val pool
PATIENCE = 10           # early stop after this many epochs without val_loss gain
ALERT = [c for c in ['falling_down', 'lying_on_the_floor', 'lying_on_floor'] if c in CLASS_NAMES]
alert_idx = [CLASS_NAMES.index(c) for c in ALERT]
save_path = os.path.join(WORK_DIR, "home_action_stgcn_best.pt")

def run_eval(model, loader, criterion):
    model.eval(); vloss = 0.0; vc = vn = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            vloss += loss.item() * y.size(0)
            vc += (out.argmax(1) == y).sum().item(); vn += y.size(0)
    return vloss / max(vn, 1), vc / max(vn, 1)

def train_one_fold(tr_items, va_items, fold):
    tr_loader, va_loader, cls_w = make_loaders(tr_items, va_items)
    model = STGCN(in_ch=3, num_classes=NUM_CLASSES).to(device)
    w = torch.tensor(cls_w / cls_w.sum() * NUM_CLASSES, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_vloss = float("inf"); best_va = 0.0; best_state = None; bad = 0
    for epoch in range(EPOCHS):
        model.train(); tc = tn = 0; run = 0.0
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    out = model(x); loss = criterion(out, y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                out = model(x); loss = criterion(out, y); loss.backward(); opt.step()
            run += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item(); tn += y.size(0)
        sched.step()
        vloss, va = run_eval(model, va_loader, criterion)
        print("  [fold %d] ep %02d/%d  loss=%.4f train_acc=%.3f  val_loss=%.4f val_acc=%.3f" % (
            fold, epoch + 1, EPOCHS, run / max(tn, 1), tc / max(tn, 1), vloss, va))
        if vloss < best_vloss - 1e-4:
            best_vloss = vloss; best_va = va; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  [fold %d] early stop @ epoch %d (best val_loss=%.4f)" % (fold, epoch + 1, best_vloss))
                break
    return best_state, best_vloss, best_va

y_pool = np.array([m["label"] for m in trainval_items])
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_vloss, fold_vacc = [], []
global_best_vloss = float("inf"); global_best_state = None
for fold, (tr_idx, va_idx) in enumerate(skf.split(trainval_items, y_pool), 1):
    tr_items = [trainval_items[i] for i in tr_idx]
    va_items = [trainval_items[i] for i in va_idx]
    print("Fold %d/%d  train=%d  val=%d" % (fold, N_SPLITS, len(tr_items), len(va_items)))
    state, vloss, va = train_one_fold(tr_items, va_items, fold)
    fold_vloss.append(vloss); fold_vacc.append(va)
    if vloss < global_best_vloss:
        global_best_vloss = vloss; global_best_state = state
        print("  ** new global best val_loss=%.4f (fold %d) **" % (vloss, fold))

print("\nCV val_loss: %.4f +/- %.4f" % (float(np.mean(fold_vloss)), float(np.std(fold_vloss))))
print("CV val_acc : %.4f +/- %.4f" % (float(np.mean(fold_vacc)), float(np.std(fold_vacc))))
torch.save({"model_state_dict": global_best_state, "class_names": CLASS_NAMES,
    "alert_class_indices": alert_idx, "val_loss": float(global_best_vloss),
    "cv_val_loss_mean": float(np.mean(fold_vloss)), "cv_val_acc_mean": float(np.mean(fold_vacc)),
    "config": {"clip_len": CLIP_LEN, "max_persons": MAX_PERSONS, "num_joints": NUM_JOINTS,
        "in_ch": 3, "img_norm": IMG_NORM, "coco_edges": COCO_EDGES,
        "pose_model": POSE_MODEL, "pose_conf": POSE_CONF}}, save_path)
print("Saved best-fold model (val_loss=%.4f) ->" % global_best_vloss, save_path)

## Evaluate - per-class precision / recall
The fall classes matter most. If their recall is low, lower their decision
threshold at inference; if precision is low, raise it / require persistence.

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(os.path.join(WORK_DIR, "home_action_stgcn_best.pt"), map_location=device)
model = STGCN(in_ch=3, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
test_loader = DataLoader(SkeletonDataset(test_items, augment=False),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(1).cpu().numpy(); y = y.numpy()
        for t, pp in zip(y, p): cm[t, pp] += 1
print("HELD-OUT TEST (25%)  -  n =", int(cm.sum()))
print("%-26s %6s %6s %6s" % ("class", "prec", "rec", "n"))
for i, c in enumerate(CLASS_NAMES):
    tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    flag = "  <-- ALERT" if i in ckpt["alert_class_indices"] else ""
    print("%-26s %6.3f %6.3f %6d%s" % (c, prec, rec, cm[i, :].sum(), flag))
print("\nTest acc:", round(float(np.trace(cm)) / max(int(cm.sum()), 1), 4),
      "| CV val_loss mean:", round(ckpt["cv_val_loss_mean"], 4))

**Held-out test:** all three official splits (train/validation/test) are pooled,
then re-split 75/25. The 25% held-out set is scored once in Evaluate for an
honest number; CV mean +/- std (printed above) tells you if it generalizes.